# Task Statement 4.5: Design Efficient Batch Processing Strategies (Message Batches API)

## 📌 Executive Overview: Goal, Scope & Exam Importance

### Certification Domain & Weight
- **Domain**: Domain 4: Prompt Engineering & Structured Output
- **Domain Weight**: 18% of total CCAR-F exam score
- **Scenario Alignment**: Scenario 5 (Continuous Integration) & Scenario 6 (Structured Data Extraction)

### The Core Goal
To design cost-optimized asynchronous data processing pipelines using the Anthropic **Message Batches API**, evaluate the architectural trade-off between the **50% token cost discount** and the **24-hour turnaround SLA**, correlate requests via **`custom_id`**, and handle partial batch failures without reprocessing successful items.

### Why This Matters in Production
Enterprise AI workloads fall into two distinct operational profiles:
1. **Interactive / Blocking Workloads**: A user waiting at a customer support chat, or an engineer waiting for a pre-merge PR review check. These require sub-minute response times and must use the standard, synchronous Messages API.
2. **Asynchronous / Non-Blocking Workloads**: Nightly repository security scans, indexing 100,000 historical PDF contracts, or backfilling embeddings.
If an architect executes a 100,000-document backfill using the synchronous Messages API:
- The organization pays 100% full list price for tokens, wasting \$10,000s in unnecessary spend.
- Synchronous calls trigger aggressive HTTP 429 rate limits and thread starvation.
By adopting the Message Batches API, the organization achieves an immediate **50% cost reduction** on all input and output tokens, while offloading rate limit queueing to Anthropic's distributed infrastructure.

### Exam Importance
Task Statement 4.5 questions test architectural cost-optimization and SLA trade-offs. The exam will ask when to select the Message Batches API versus the synchronous API, how to track requests using `custom_id`, and how to recover when a small fraction of batch items fail.

---

## 📖 Deep Narrative: Why We Do What We Do

### 1. The Cost vs. Latency Trade-Off Matrix
The Message Batches API introduces a fundamental architectural contract:
- **The Benefit**: **50% discount** on all token pricing (both prompt inputs and completion outputs).
- **The Constraint**: Results are delivered asynchronously within a window of up to **24 hours** (though many finish much faster).
- **The Rule of Thumb**:
  - If a human or CI blocker is waiting $\rightarrow$ **Synchronous Messages API**.
  - If the job runs overnight, in background queues, or on non-blocking backfills $\rightarrow$ **Message Batches API**.

### 2. Request Correlation via `custom_id`
Because batch processing is asynchronous, thousands of requests run concurrently across distributed clusters. Results do not necessarily finish in the order they were submitted:
- Every request payload must declare a unique, application-controlled identifier:
  ```json
  {
    "custom_id": "inv_rec_2025_99812",
    "params": {
      "model": "claude-sonnet-4-5",
      "max_tokens": 1024,
      "messages": [{"role": "user", "content": "..."}]
    }
  }
  ```
- When the batch finishes and results are streamed, each result record includes its matching `custom_id`. This allows the host application to write extracted facts directly into the correct database row with zero index-shifting errors.

### 3. Partial Failure Recovery Protocol
In a batch of 10,000 documents, 9,995 may succeed while 5 fail due to transient carrier timeouts:
- **Anti-Pattern**: Re-submitting the entire 10,000-item batch (wastes tokens and creates duplicate records).
- **Production Standard**: Inspect individual item statuses in the results stream. Filter for `result.type == "errored"`, extract only those 5 `custom_id`s, and compile a small targeted retry batch.

---

## 🔑 Key Technical Terminology Explained

- **Message Batches API**: Anthropic's asynchronous batch processing endpoint (`/v1/messages/batches`) designed for high-volume, non-time-critical workloads with a 50% token cost reduction.
- **50% Token Discount**: The discounted pricing applied automatically to all input tokens, output tokens, and cache writes/reads executed through the Message Batches API.
- **`custom_id`**: A mandatory client-provided string attached to each batch item, used to correlate asynchronous results with external database records.
- **Batch Processing Status**: The lifecycle states of a batch: `in_progress` (processing), `canceling`, `ended` (completed).
- **Targeted Partial Resubmission**: The recovery practice of isolating only the specific `custom_id` items that experienced errors and submitting a minimal follow-up batch.

---

## 🏗️ Architecture: Asynchronous Message Batches Lifecycle

```mermaid
flowchart TD
    Client["Application Ingestion Engine"] --> BuildBatch["Construct Batch Payload<br>(List of requests with unique custom_id)"]
    
    BuildBatch --> SubmitAPI["Submit to Message Batches API<br>client.messages.batches.create(...)"]
    
    SubmitAPI --> StatusCheck{"Poll Batch Status<br>client.messages.batches.retrieve(...)"}
    
    StatusCheck -->|"processing_status: in_progress"| SleepPoll["Wait with exponential backoff"]
    SleepPoll --> StatusCheck
    
    StatusCheck -->|"processing_status: ended"| StreamResults["Stream Results File<br>client.messages.batches.results(...)"]
    
    StreamResults --> InspectResults{"Evaluate Item Results"}
    
    InspectResults -->|"result.type: succeeded"| SaveDB["Store Extracted Record in DB<br>(Keyed by custom_id)"]
    InspectResults -->|"result.type: errored"| RetryQueue["Enqueue only failed custom_ids<br>into targeted retry batch"]
```

---

## ⚠️ Critical Exam Distinctions & Distractor Teardown

| Dimension | Message Batches API | Standard Messages API | Exam Trap to Reject |
|---|---|---|---|
| **Pricing** | **50% cost discount** on all tokens. | Standard list pricing. | **TRAP**: Using synchronous calls for massive non-urgent backfills, wasting 50% of budget. |
| **SLA / Turnaround** | Up to **24 hours** (asynchronous). | Real-time (seconds). | **TRAP**: Selecting Message Batches for interactive developer tooling or blocking PR checks. |
| **Request Correlation** | Correlate results using **`custom_id`**. | Immediate variable assignment. | **TRAP**: Assuming batch results return in the identical order they were submitted. |
| **Failure Recovery** | **Retry only failed `custom_id` items**. | Retry individual API call. | **TRAP**: Resubmitting the entire batch of 10,000 items when only 5 experienced errors. |

---

## 💡 Architectural Decision Matrix: API Selection

| Workload Scenario | Optimal API | SLA Requirement | Cost Benefit |
|---|---|---|---|
| **Blocking Pre-Merge PR Review** | Standard Messages API | < 3 minutes | Real-time feedback |
| **Overnight Full Repo Audit** | Message Batches API | 24 hours | 50% Token Discount |
| **Live Customer Chatbot** | Standard Messages API | < 2.5 seconds | Streaming real-time |
| **Backfilling 100k Legal Contracts** | Message Batches API | 24 hours | 50% Token Discount |

---

## 🧠 Claude 4.5 Extended Thinking & Parameter Standards

- **Extended Thinking in Batches**: Message Batches fully support Extended Thinking (`budget_tokens`). The 50% discount applies to thinking tokens as well!
- **Sampling Rules**: Omit `temperature` whenever thinking is enabled in batch payloads.
- **Batch Payload Uniformity**: Ensure all items in the batch omit `temperature` when thinking is enabled to prevent individual item rejections.

---

## 🛠️ Hands-on Workbook Guide: Following the Code Step-by-Step

When exploring [`ts_4_5_message_batches_api.ipynb`](ts_4_5_message_batches_api.ipynb):
1. **Step 1: Setup & Batch Simulator**: Initializes the offline Message Batches lifecycle simulator.
2. **Step 2: Constructing Batch Payloads**: Builds batch payloads with unique `custom_id` identifiers.
3. **Step 3: Simulating Batch Ingestion & Results**: Demonstrates polling batch status and streaming results.
4. **Step 4: Partial Failure Resubmission**: Demonstrates filtering failed items and resubmitting only affected `custom_id`s.
5. **Step 5: Verification Assertions**: Verifies that 100% of records correlate with database rows and zero duplicates occur.

In [ ]:
# Step 1: Environment & Setup
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(" Environment initialized for TS 4.5 Message Batches API.")

### 📦 Step 2: Message Batches Engine Implementation
We implement the batch management engine with partial failure isolation and cost savings calculations.

In [ ]:
# Step 2: Batch Engine Logic
@dataclass
class BatchRequest:
    custom_id: str
    params: Dict[str, Any]


@dataclass
class BatchResponse:
    batch_id: str
    status: str
    total_requests: int
    succeeded: List[str]
    failed: List[str]


class MessageBatchesEngine:
    def __init__(self) -> None:
        self.batches: Dict[str, BatchResponse] = {}

    def submit_batch(self, requests: List[BatchRequest]) -> str:
        batch_id = f"batch_{len(self.batches) + 1:03d}"
        succeeded = []
        failed = []

        # Simulate batch processing
        for req in requests:
            # Simulate failure for requests over token limit
            if req.params.get("token_estimate", 0) > 100_000:
                failed.append(req.custom_id)
            else:
                succeeded.append(req.custom_id)

        self.batches[batch_id] = BatchResponse(
            batch_id=batch_id,
            status="ended",
            total_requests=len(requests),
            succeeded=succeeded,
            failed=failed,
        )
        return batch_id

    def filter_failed_for_resubmission(self, batch_id: str) -> List[str]:
        """Extracts failed custom_ids to prevent re-running already successful requests."""
        batch = self.batches.get(batch_id)
        return batch.failed if batch else []

    @staticmethod
    def calculate_cost_savings(total_tokens: int, base_cost_per_million: float = 3.0) -> Dict[str, float]:
        """Message Batches API provides an exact 50% discount."""
        standard_cost = (total_tokens / 1_000_000) * base_cost_per_million
        batch_cost = standard_cost * 0.50
        savings = standard_cost - batch_cost
        return {
            "standard_cost": round(standard_cost, 4),
            "batch_cost": round(batch_cost, 4),
            "savings": round(savings, 4),
            "discount_pct": 50.0,
        }

print(" MessageBatchesEngine initialized.")

### 🧪 Step 3: Interactive Simulation
1. Submit batch of requests.
2. Filter failed custom_ids for partial resubmission.
3. Calculate cost savings on 50M tokens.

In [ ]:
# Step 3: Simulation
engine = MessageBatchesEngine()

# 1. Submit batch with mixed token estimates
requests = [
    BatchRequest(custom_id="doc_001", params={"token_estimate": 2500}),
    BatchRequest(custom_id="doc_002", params={"token_estimate": 150_000}),  # Fails!
    BatchRequest(custom_id="doc_003", params={"token_estimate": 4000}),
]
batch_id = engine.submit_batch(requests)
print(f"Batch submitted: {batch_id}")

# 2. Extract failed items for resubmission
failed_ids = engine.filter_failed_for_resubmission(batch_id)
print("Failed requests needing resubmission:", failed_ids)

# 3. Calculate 50% cost savings on 50M tokens
savings = MessageBatchesEngine.calculate_cost_savings(50_000_000)
print("\nCost Analysis on 50M Tokens:", savings)

### 🔬 Step 4: Verification Assertions
We assert that:
1. Succeeded and failed custom_ids are partitioned.
2. Resubmission selects only failed IDs.
3. Batch cost savings calculate to exactly 50%.

In [ ]:
# Step 4: Verification Checks
assert failed_ids == ["doc_002"]
assert savings["discount_pct"] == 50.0
assert savings["batch_cost"] == savings["standard_cost"] / 2

print(" All verification assertions passed! Message Batches API verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Use the **Message Batches API** for non-real-time jobs to capture the **50% cost reduction**.
- **Rule 2**: The turnaround SLA is **24 hours**.
- **Rule 3**: On partial batch failure, resubmit **only the failed custom_ids**; never re-run the entire batch.